# Experiment 05: Seizure Event-Level Evaluation

## Objective

The previous experiments evaluated the EEG seizure detection model at the
individual window level.

This experiment evaluates the model at the seizure-event and
recording level.

The purpose is to determine whether the model can successfully detect
seizure events within complete EEG recordings.

## Evaluation Strategy

1. Generate seizure probabilities for EEG windows from unseen recordings.
2. Apply the candidate classification threshold identified in Experiment 04.
3. Group window predictions according to their original EEG recording.
4. Identify recordings containing annotated seizure windows.
5. Determine whether the model detected at least one seizure window
   within each seizure-containing recording.
6. Calculate recording-level seizure detection sensitivity.
7. Analyze detected and missed seizure-containing recordings.

## Research Question

Can the trained EEG seizure detection model detect seizure events
within complete unseen EEG recordings?

## Important Note

This experiment is part of a research and learning prototype and is
not a clinically validated diagnostic system.

Recording-level detection is different from window-level detection.
A model may detect some seizure windows but still fail to detect an
entire seizure event reliably.

In [1]:
# ============================================================
# CELL 2: IMPORT LIBRARIES
# ============================================================

import os
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ============================================================
# CELL 3: DEFINE PROJECT PATHS
# ============================================================

# Project root directory
PROJECT_ROOT = Path(
    r"C:\Users\Prajapati_Shivam\EEG-Seizure-Detection"
)

# Data directory
DATA_DIR = PROJECT_ROOT / "data"

# Results directory
RESULTS_DIR = PROJECT_ROOT / "results"

# Model directory
MODELS_DIR = PROJECT_ROOT / "models"

# Required files
FEATURES_PATH = DATA_DIR / "features.csv"

METADATA_PATH = DATA_DIR / "window_metadata.csv"

# Class-weighted Random Forest model
MODEL_PATH = (
    MODELS_DIR
    / "random_forest_class_weighted.pkl"
)

print("Project Root:")
print(PROJECT_ROOT)

print("\nFeatures Path:")
print(FEATURES_PATH)

print("\nMetadata Path:")
print(METADATA_PATH)

print("\nModel Path:")
print(MODEL_PATH)

Project Root:
C:\Users\Prajapati_Shivam\EEG-Seizure-Detection

Features Path:
C:\Users\Prajapati_Shivam\EEG-Seizure-Detection\data\features.csv

Metadata Path:
C:\Users\Prajapati_Shivam\EEG-Seizure-Detection\data\window_metadata.csv

Model Path:
C:\Users\Prajapati_Shivam\EEG-Seizure-Detection\models\random_forest_class_weighted.pkl


In [3]:
# ============================================================
# CELL 4: VERIFY REQUIRED FILES
# ============================================================

print("=" * 60)
print("CHECKING REQUIRED FILES")
print("=" * 60)

required_files = {
    "features.csv": FEATURES_PATH,
    "window_metadata.csv": METADATA_PATH,
    "random_forest_class_weighted.pkl": MODEL_PATH
}

all_files_found = True

for file_name, file_path in required_files.items():

    if file_path.exists():
        print(f"Found: {file_name}")
    else:
        print(f"ERROR: Missing {file_name}")
        print(f"Expected location: {file_path}")
        all_files_found = False

if all_files_found:
    print("\nAll required files found successfully.")
else:
    print("\nERROR: One or more required files are missing.")

CHECKING REQUIRED FILES
Found: features.csv
Found: window_metadata.csv
Found: random_forest_class_weighted.pkl

All required files found successfully.


In [4]:
# ============================================================
# CELL 5: LOAD FEATURES, METADATA, AND MODEL
# ============================================================

print("=" * 60)
print("LOADING FEATURES, METADATA, AND MODEL")
print("=" * 60)

# Load feature dataset
features_df = pd.read_csv(
    FEATURES_PATH
)

# Load window metadata
metadata_df = pd.read_csv(
    METADATA_PATH
)

# Load class-weighted Random Forest
model = joblib.load(
    MODEL_PATH
)

print("\nFeature Dataset Shape:")
print(features_df.shape)

print("\nMetadata Shape:")
print(metadata_df.shape)

print("\nModel:")
print(model)

print("\nFeature Columns:")
print(features_df.columns.tolist())

print("\nMetadata Columns:")
print(metadata_df.columns.tolist())

LOADING FEATURES, METADATA, AND MODEL

Feature Dataset Shape:
(13181, 9)

Metadata Shape:
(13181, 4)

Model:
RandomForestClassifier(class_weight='balanced', n_estimators=200, n_jobs=-1,
                       random_state=42)

Feature Columns:
['Mean', 'Std', 'Variance', 'Delta', 'Theta', 'Alpha', 'Beta', 'Gamma', 'Label']

Metadata Columns:
['file', 'window_start', 'window_end', 'label']


In [5]:
# ============================================================
# CELL 6: VERIFY DATASET ALIGNMENT
# ============================================================

print("=" * 60)
print("VERIFYING DATASET ALIGNMENT")
print("=" * 60)

# Check number of rows
features_count = len(features_df)
metadata_count = len(metadata_df)

print("\nFeature rows:", features_count)
print("Metadata rows:", metadata_count)

# Check that both datasets have the same number of windows
if features_count == metadata_count:

    print("\nSUCCESS: Feature and metadata row counts match.")

else:

    print("\nERROR: Feature and metadata row counts do NOT match.")

    raise ValueError(
        "Feature and metadata datasets are not aligned."
    )


# Check labels between features and metadata
feature_labels = features_df["Label"].to_numpy()
metadata_labels = metadata_df["label"].to_numpy()

# Compare labels row by row
labels_match = np.array_equal(
    feature_labels,
    metadata_labels
)

print("\nLabels match:", labels_match)

if labels_match:

    print(
        "SUCCESS: Feature labels and metadata labels "
        "are perfectly aligned."
    )

else:

    print(
        "ERROR: Feature labels and metadata labels "
        "do NOT match."
    )

    raise ValueError(
        "Feature labels and metadata labels are not aligned."
    )


# Display first few rows for visual verification
print("\nFirst 5 feature labels:")
print(feature_labels[:5])

print("\nFirst 5 metadata labels:")
print(metadata_labels[:5])

print("\nDataset alignment verification completed successfully.")

VERIFYING DATASET ALIGNMENT

Feature rows: 13181
Metadata rows: 13181

SUCCESS: Feature and metadata row counts match.

Labels match: True
SUCCESS: Feature labels and metadata labels are perfectly aligned.

First 5 feature labels:
[0 0 0 0 0]

First 5 metadata labels:
[0 0 0 0 0]

Dataset alignment verification completed successfully.


In [6]:
# ============================================================
# CELL 7: GENERATE WINDOW-LEVEL PREDICTIONS
# ============================================================

print("=" * 60)
print("GENERATING WINDOW-LEVEL PREDICTIONS")
print("=" * 60)

# ------------------------------------------------------------
# Define feature columns
# ------------------------------------------------------------

feature_columns = [
    "Mean",
    "Std",
    "Variance",
    "Delta",
    "Theta",
    "Alpha",
    "Beta",
    "Gamma"
]

# Extract feature matrix
X = features_df[feature_columns]

# Extract true labels
y_true = features_df["Label"].to_numpy()

# ------------------------------------------------------------
# Generate seizure probabilities
# ------------------------------------------------------------

seizure_probabilities = model.predict_proba(X)[:, 1]

print("\nProbability predictions generated successfully.")

print("\nNumber of windows:")
print(len(seizure_probabilities))

print("\nProbability range:")
print(
    "Minimum:",
    f"{seizure_probabilities.min():.6f}"
)

print(
    "Maximum:",
    f"{seizure_probabilities.max():.6f}"
)

# ------------------------------------------------------------
# Apply candidate threshold from Experiment 04
# ------------------------------------------------------------

candidate_threshold = 0.10

window_predictions = (
    seizure_probabilities >= candidate_threshold
).astype(int)

print("\nCandidate Threshold:")
print(candidate_threshold)

print("\nPredicted class distribution:")

print(
    pd.Series(
        window_predictions
    ).value_counts().sort_index()
)

# ------------------------------------------------------------
# Add predictions to metadata
# ------------------------------------------------------------

evaluation_df = metadata_df.copy()

evaluation_df["true_label"] = y_true

evaluation_df["seizure_probability"] = (
    seizure_probabilities
)

evaluation_df["predicted_label"] = (
    window_predictions
)

print("\nEvaluation dataset created successfully.")

print("\nEvaluation Dataset Shape:")
print(evaluation_df.shape)

print("\nFirst 5 rows:")
print(
    evaluation_df.head()
)

GENERATING WINDOW-LEVEL PREDICTIONS

Probability predictions generated successfully.

Number of windows:
13181

Probability range:
Minimum: 0.000000
Maximum: 0.975000

Candidate Threshold:
0.1

Predicted class distribution:
0    13050
1      131
Name: count, dtype: int64

Evaluation dataset created successfully.

Evaluation Dataset Shape:
(13181, 7)

First 5 rows:
           file  window_start  window_end  label  true_label  \
0  chb01_01.edf           0.0         4.0      0           0   
1  chb01_01.edf           4.0         8.0      0           0   
2  chb01_01.edf           8.0        12.0      0           0   
3  chb01_01.edf          12.0        16.0      0           0   
4  chb01_01.edf          16.0        20.0      0           0   

   seizure_probability  predicted_label  
0                  0.0                0  
1                  0.0                0  
2                  0.0                0  
3                  0.0                0  
4                  0.0                

In [7]:
# ============================================================
# CELL 8: SELECT UNSEEN TEST RECORDINGS
# ============================================================

print("=" * 60)
print("SELECTING UNSEEN TEST RECORDINGS")
print("=" * 60)

# These are the same unseen recordings used in
# Experiment 03 and Experiment 04.

test_recordings = [
    "chb01_18.edf",
    "chb01_21.edf",
    "chb01_26.edf",
    "chb01_42.edf",
    "chb01_46.edf"
]

print("\nUnseen test recordings:")

for recording in test_recordings:
    print(" TEST:", recording)

# Select only unseen test recordings
test_evaluation_df = evaluation_df[
    evaluation_df["file"].isin(test_recordings)
].copy()

# Reset index
test_evaluation_df = test_evaluation_df.reset_index(
    drop=True
)

print("\nNumber of unseen test recordings:")
print(
    test_evaluation_df["file"].nunique()
)

print("\nTesting Dataset Shape:")
print(
    test_evaluation_df.shape
)

print("\nTest recordings found in dataset:")

print(
    test_evaluation_df["file"].unique()
)

# Verify that only the expected recordings are present
actual_test_recordings = set(
    test_evaluation_df["file"].unique()
)

expected_test_recordings = set(
    test_recordings
)

if actual_test_recordings == expected_test_recordings:

    print(
        "\nSUCCESS: All expected unseen test recordings "
        "were selected."
    )

else:

    print(
        "\nERROR: Test recording selection mismatch."
    )

    print(
        "Expected:",
        expected_test_recordings
    )

    print(
        "Found:",
        actual_test_recordings
    )

    raise ValueError(
        "Unseen test recording selection is incorrect."
    )

SELECTING UNSEEN TEST RECORDINGS

Unseen test recordings:
 TEST: chb01_18.edf
 TEST: chb01_21.edf
 TEST: chb01_26.edf
 TEST: chb01_42.edf
 TEST: chb01_46.edf

Number of unseen test recordings:
5

Testing Dataset Shape:
(4181, 7)

Test recordings found in dataset:
['chb01_18.edf' 'chb01_21.edf' 'chb01_26.edf' 'chb01_42.edf'
 'chb01_46.edf']

SUCCESS: All expected unseen test recordings were selected.


In [8]:
# ============================================================
# CELL 9: RECORDING-LEVEL GROUND TRUTH SUMMARY
# ============================================================

print("=" * 60)
print("RECORDING-LEVEL GROUND TRUTH SUMMARY")
print("=" * 60)

# Group test windows by recording
recording_ground_truth = (
    test_evaluation_df
    .groupby("file")
    .agg(
        total_windows=("true_label", "count"),
        seizure_windows=("true_label", "sum"),
        normal_windows=(
            "true_label",
            lambda x: (x == 0).sum()
        )
    )
    .reset_index()
)

# Determine whether each recording contains a seizure
recording_ground_truth["actual_seizure_recording"] = (
    recording_ground_truth["seizure_windows"] > 0
).astype(int)

print("\nRecording-Level Ground Truth:")
print(
    recording_ground_truth.to_string(
        index=False
    )
)

print("\n============================================================")
print("SEIZURE-CONTAINING TEST RECORDINGS")
print("============================================================")

seizure_test_recordings = (
    recording_ground_truth[
        recording_ground_truth[
            "actual_seizure_recording"
        ] == 1
    ]["file"]
    .tolist()
)

for recording in seizure_test_recordings:
    print(
        "SEIZURE RECORDING:",
        recording
    )

print(
    "\nNumber of seizure-containing test recordings:",
    len(seizure_test_recordings)
)

print("\n============================================================")
print("NORMAL-ONLY TEST RECORDINGS")
print("============================================================")

normal_test_recordings = (
    recording_ground_truth[
        recording_ground_truth[
            "actual_seizure_recording"
        ] == 0
    ]["file"]
    .tolist()
)

for recording in normal_test_recordings:
    print(
        "NORMAL-ONLY RECORDING:",
        recording
    )

print(
    "\nNumber of normal-only test recordings:",
    len(normal_test_recordings)
)

RECORDING-LEVEL GROUND TRUTH SUMMARY

Recording-Level Ground Truth:
        file  total_windows  seizure_windows  normal_windows  actual_seizure_recording
chb01_18.edf            900               23             877                         1
chb01_21.edf            900               24             876                         1
chb01_26.edf            581               26             555                         1
chb01_42.edf            900                0             900                         0
chb01_46.edf            900                0             900                         0

SEIZURE-CONTAINING TEST RECORDINGS
SEIZURE RECORDING: chb01_18.edf
SEIZURE RECORDING: chb01_21.edf
SEIZURE RECORDING: chb01_26.edf

Number of seizure-containing test recordings: 3

NORMAL-ONLY TEST RECORDINGS
NORMAL-ONLY RECORDING: chb01_42.edf
NORMAL-ONLY RECORDING: chb01_46.edf

Number of normal-only test recordings: 2


In [9]:
# ============================================================
# CELL 10: RECORDING-LEVEL SEIZURE DETECTION
# ============================================================

print("=" * 60)
print("RECORDING-LEVEL SEIZURE DETECTION")
print("=" * 60)

# A recording is considered detected if the model predicts
# at least one seizure window in that recording.

recording_predictions = (
    test_evaluation_df
    .groupby("file")
    .agg(
        total_windows=("true_label", "count"),

        actual_seizure_windows=(
            "true_label",
            "sum"
        ),

        predicted_seizure_windows=(
            "predicted_label",
            "sum"
        ),

        maximum_seizure_probability=(
            "seizure_probability",
            "max"
        )
    )
    .reset_index()
)

# Determine actual recording-level seizure status
recording_predictions["actual_seizure_recording"] = (
    recording_predictions[
        "actual_seizure_windows"
    ] > 0
).astype(int)

# Determine predicted recording-level seizure status
# If at least one window is predicted as seizure,
# the recording is considered detected.
recording_predictions["predicted_seizure_recording"] = (
    recording_predictions[
        "predicted_seizure_windows"
    ] > 0
).astype(int)

# Determine whether the recording was correctly detected
recording_predictions["correct_detection"] = (
    recording_predictions[
        "actual_seizure_recording"
    ]
    ==
    recording_predictions[
        "predicted_seizure_recording"
    ]
).astype(int)

print("\nRecording-Level Prediction Results:")
print(
    recording_predictions.to_string(
        index=False
    )
)

print("\n============================================================")
print("SEIZURE RECORDING DETECTION RESULTS")
print("============================================================")

# Analyze seizure-containing recordings only
seizure_recording_results = (
    recording_predictions[
        recording_predictions[
            "actual_seizure_recording"
        ] == 1
    ]
)

for _, row in seizure_recording_results.iterrows():

    if row["predicted_seizure_recording"] == 1:

        status = "DETECTED"

    else:

        status = "MISSED"

    print(
        f"{row['file']}: {status}"
    )

    print(
        "  Actual seizure windows:",
        int(row["actual_seizure_windows"])
    )

    print(
        "  Predicted seizure windows:",
        int(row["predicted_seizure_windows"])
    )

    print(
        "  Maximum seizure probability:",
        f"{row['maximum_seizure_probability']:.4f}"
    )

print("\nRecording-level detection analysis completed.")

RECORDING-LEVEL SEIZURE DETECTION

Recording-Level Prediction Results:
        file  total_windows  actual_seizure_windows  predicted_seizure_windows  maximum_seizure_probability  actual_seizure_recording  predicted_seizure_recording  correct_detection
chb01_18.edf            900                      23                         17                        0.930                         1                            1                  1
chb01_21.edf            900                      24                         28                        0.930                         1                            1                  1
chb01_26.edf            581                      26                         25                        0.880                         1                            1                  1
chb01_42.edf            900                       0                         21                        0.525                         0                            1                  0
chb01_46.edf       

In [10]:
# ============================================================
# CELL 11: RECORDING-LEVEL METRICS
# ============================================================

print("=" * 60)
print("RECORDING-LEVEL EVALUATION METRICS")
print("=" * 60)

# Actual recording-level labels
y_recording_true = (
    recording_predictions[
        "actual_seizure_recording"
    ].to_numpy()
)

# Predicted recording-level labels
y_recording_pred = (
    recording_predictions[
        "predicted_seizure_recording"
    ].to_numpy()
)

# ------------------------------------------------------------
# Confusion Matrix
# ------------------------------------------------------------

cm_recording = confusion_matrix(
    y_recording_true,
    y_recording_pred,
    labels=[0, 1]
)

tn_r, fp_r, fn_r, tp_r = (
    cm_recording.ravel()
)

print("\nConfusion Matrix:")
print(cm_recording)

print("\nConfusion Matrix Components:")
print("True Negatives :", tn_r)
print("False Positives:", fp_r)
print("False Negatives:", fn_r)
print("True Positives :", tp_r)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

accuracy_recording = accuracy_score(
    y_recording_true,
    y_recording_pred
)

precision_recording = precision_score(
    y_recording_true,
    y_recording_pred,
    zero_division=0
)

sensitivity_recording = recall_score(
    y_recording_true,
    y_recording_pred,
    zero_division=0
)

specificity_recording = (
    tn_r / (tn_r + fp_r)
    if (tn_r + fp_r) > 0
    else 0
)

f1_recording = f1_score(
    y_recording_true,
    y_recording_pred,
    zero_division=0
)

print("\n============================================================")
print("RECORDING-LEVEL METRICS")
print("============================================================")

print(
    f"Accuracy:           {accuracy_recording:.4f}"
)

print(
    f"Precision:          {precision_recording:.4f}"
)

print(
    f"Sensitivity/Recall: {sensitivity_recording:.4f}"
)

print(
    f"Specificity:        {specificity_recording:.4f}"
)

print(
    f"F1-Score:           {f1_recording:.4f}"
)

print("\nRecording-level evaluation completed.")

RECORDING-LEVEL EVALUATION METRICS

Confusion Matrix:
[[0 2]
 [0 3]]

Confusion Matrix Components:
True Negatives : 0
False Positives: 2
False Negatives: 0
True Positives : 3

RECORDING-LEVEL METRICS
Accuracy:           0.6000
Precision:          0.6000
Sensitivity/Recall: 1.0000
Specificity:        0.0000
F1-Score:           0.7500

Recording-level evaluation completed.


In [11]:
# ============================================================
# CELL 12: SEIZURE WINDOW OVERLAP ANALYSIS
# ============================================================

print("=" * 60)
print("SEIZURE WINDOW OVERLAP ANALYSIS")
print("=" * 60)

# ------------------------------------------------------------
# Analyze only recordings that actually contain seizures
# ------------------------------------------------------------

seizure_recording_list = (
    recording_ground_truth[
        recording_ground_truth[
            "actual_seizure_recording"
        ] == 1
    ]["file"]
    .tolist()
)

event_results = []

for recording in seizure_recording_list:

    # Get all windows for this recording
    recording_data = test_evaluation_df[
        test_evaluation_df["file"] == recording
    ].copy()

    # Actual seizure windows
    actual_seizure_data = recording_data[
        recording_data["true_label"] == 1
    ]

    # Predicted seizure windows
    predicted_seizure_data = recording_data[
        recording_data["predicted_label"] == 1
    ]

    # --------------------------------------------------------
    # Check prediction overlap with actual seizure windows
    # --------------------------------------------------------

    overlapping_predictions = []

    for _, pred_row in predicted_seizure_data.iterrows():

        pred_start = pred_row["window_start"]
        pred_end = pred_row["window_end"]

        overlaps_actual_seizure = False

        for _, actual_row in actual_seizure_data.iterrows():

            actual_start = actual_row["window_start"]
            actual_end = actual_row["window_end"]

            # Check whether the predicted window overlaps
            # an actual seizure window
            if (
                pred_start < actual_end
                and
                pred_end > actual_start
            ):

                overlaps_actual_seizure = True
                break

        overlapping_predictions.append(
            overlaps_actual_seizure
        )

    # Count overlapping and non-overlapping predictions
    overlapping_count = sum(
        overlapping_predictions
    )

    non_overlapping_count = (
        len(overlapping_predictions)
        - overlapping_count
    )

    # --------------------------------------------------------
    # Determine event detection
    # --------------------------------------------------------

    if overlapping_count > 0:

        event_detected = 1

    else:

        event_detected = 0

    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------

    event_results.append({

        "file": recording,

        "actual_seizure_windows":
            len(actual_seizure_data),

        "predicted_seizure_windows":
            len(predicted_seizure_data),

        "overlapping_predictions":
            overlapping_count,

        "non_overlapping_predictions":
            non_overlapping_count,

        "maximum_seizure_probability":
            recording_data[
                "seizure_probability"
            ].max(),

        "event_detected":
            event_detected
    })


# Convert results to DataFrame
event_results_df = pd.DataFrame(
    event_results
)

print("\n============================================================")
print("SEIZURE EVENT OVERLAP RESULTS")
print("============================================================")

print(
    event_results_df.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

total_events = len(
    event_results_df
)

detected_events = (
    event_results_df[
        "event_detected"
    ].sum()
)

missed_events = (
    total_events
    - detected_events
)

event_sensitivity = (
    detected_events / total_events
    if total_events > 0
    else 0
)

print("\n============================================================")
print("EVENT-LEVEL SUMMARY")
print("============================================================")

print(
    "Total seizure-containing recordings:",
    total_events
)

print(
    "Detected seizure events:",
    detected_events
)

print(
    "Missed seizure events:",
    missed_events
)

print(
    f"Event-level sensitivity: "
    f"{event_sensitivity:.4f}"
)

print(
    f"Event-level sensitivity (%): "
    f"{event_sensitivity * 100:.2f}%"
)

SEIZURE WINDOW OVERLAP ANALYSIS

SEIZURE EVENT OVERLAP RESULTS
        file  actual_seizure_windows  predicted_seizure_windows  overlapping_predictions  non_overlapping_predictions  maximum_seizure_probability  event_detected
chb01_18.edf                      23                         17                       17                            0                         0.93               1
chb01_21.edf                      24                         28                       22                            6                         0.93               1
chb01_26.edf                      26                         25                       24                            1                         0.88               1

EVENT-LEVEL SUMMARY
Total seizure-containing recordings: 3
Detected seizure events: 3
Missed seizure events: 0
Event-level sensitivity: 1.0000
Event-level sensitivity (%): 100.00%


In [12]:
# ============================================================
# CELL 13: CONSECUTIVE-WINDOW EVENT DETECTION ANALYSIS
# ============================================================

print("=" * 60)
print("CONSECUTIVE-WINDOW EVENT DETECTION ANALYSIS")
print("=" * 60)

# ------------------------------------------------------------
# Define consecutive-window criteria
# ------------------------------------------------------------

consecutive_requirements = [
    1,
    2,
    3,
    5
]

# Store results
consecutive_results = []

# ------------------------------------------------------------
# Analyze each consecutive-window requirement
# ------------------------------------------------------------

for required_consecutive in consecutive_requirements:

    print("\n" + "=" * 60)

    print(
        f"REQUIRED CONSECUTIVE POSITIVE WINDOWS: "
        f"{required_consecutive}"
    )

    print("=" * 60)

    recording_results = []

    # --------------------------------------------------------
    # Analyze every test recording
    # --------------------------------------------------------

    for recording in test_recordings:

        recording_data = test_evaluation_df[
            test_evaluation_df["file"] == recording
        ].copy()

        # Ensure windows are sorted chronologically
        recording_data = recording_data.sort_values(
            "window_start"
        ).reset_index(drop=True)

        # Binary prediction sequence
        predictions = (
            recording_data[
                "predicted_label"
            ]
            .to_numpy()
        )

        # ----------------------------------------------------
        # Find longest consecutive positive sequence
        # ----------------------------------------------------

        longest_run = 0
        current_run = 0

        for prediction in predictions:

            if prediction == 1:

                current_run += 1

                longest_run = max(
                    longest_run,
                    current_run
                )

            else:

                current_run = 0

        # ----------------------------------------------------
        # Determine recording-level prediction
        # ----------------------------------------------------

        predicted_seizure_recording = int(
            longest_run >= required_consecutive
        )

        # Actual recording-level label
        actual_seizure_recording = int(
            recording_data[
                "true_label"
            ].sum() > 0
        )

        # Store recording result
        recording_results.append({

            "file": recording,

            "actual_seizure_recording":
                actual_seizure_recording,

            "longest_positive_run":
                longest_run,

            "predicted_seizure_recording":
                predicted_seizure_recording

        })

    # --------------------------------------------------------
    # Convert to DataFrame
    # --------------------------------------------------------

    recording_results_df = pd.DataFrame(
        recording_results
    )

    # --------------------------------------------------------
    # Calculate confusion matrix
    # --------------------------------------------------------

    y_true_recording = (
        recording_results_df[
            "actual_seizure_recording"
        ]
        .to_numpy()
    )

    y_pred_recording = (
        recording_results_df[
            "predicted_seizure_recording"
        ]
        .to_numpy()
    )

    cm = confusion_matrix(
        y_true_recording,
        y_pred_recording,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    # --------------------------------------------------------
    # Calculate metrics
    # --------------------------------------------------------

    sensitivity = recall_score(
        y_true_recording,
        y_pred_recording,
        zero_division=0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0
    )

    precision = precision_score(
        y_true_recording,
        y_pred_recording,
        zero_division=0
    )

    accuracy = accuracy_score(
        y_true_recording,
        y_pred_recording
    )

    f1 = f1_score(
        y_true_recording,
        y_pred_recording,
        zero_division=0
    )

    # --------------------------------------------------------
    # Store summary
    # --------------------------------------------------------

    consecutive_results.append({

        "consecutive_windows":
            required_consecutive,

        "time_seconds":
            required_consecutive * 4,

        "accuracy":
            accuracy,

        "precision":
            precision,

        "sensitivity":
            sensitivity,

        "specificity":
            specificity,

        "f1_score":
            f1,

        "TN":
            tn,

        "FP":
            fp,

        "FN":
            fn,

        "TP":
            tp
    })

    # --------------------------------------------------------
    # Print recording-level details
    # --------------------------------------------------------

    print(
        recording_results_df.to_string(
            index=False
        )
    )

    print("\nConfusion Matrix:")
    print(cm)

    print(
        f"\nSensitivity: {sensitivity:.4f}"
    )

    print(
        f"Specificity: {specificity:.4f}"
    )


# ============================================================
# FINAL COMPARISON
# ============================================================

consecutive_results_df = pd.DataFrame(
    consecutive_results
)

print("\n" + "=" * 60)
print("CONSECUTIVE-WINDOW EVENT DETECTION COMPARISON")
print("=" * 60)

print(
    consecutive_results_df.to_string(
        index=False
    )
)

CONSECUTIVE-WINDOW EVENT DETECTION ANALYSIS

REQUIRED CONSECUTIVE POSITIVE WINDOWS: 1
        file  actual_seizure_recording  longest_positive_run  predicted_seizure_recording
chb01_18.edf                         1                    15                            1
chb01_21.edf                         1                    22                            1
chb01_26.edf                         1                    24                            1
chb01_42.edf                         0                     3                            1
chb01_46.edf                         0                     1                            1

Confusion Matrix:
[[0 2]
 [0 3]]

Sensitivity: 1.0000
Specificity: 0.0000

REQUIRED CONSECUTIVE POSITIVE WINDOWS: 2
        file  actual_seizure_recording  longest_positive_run  predicted_seizure_recording
chb01_18.edf                         1                    15                            1
chb01_21.edf                         1                    22                 